In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession



In [2]:
load_dotenv(override=True)

True

In [3]:
# Create an agent
agent = Agent(name="Jokester", 
              instructions="You're a talanted joke teller", 
              model="gpt-5.4-mini") 

In [4]:
# Run the agent with user prompt
result = await Runner.run(agent, "Tell a joke about norwegians")

In [5]:
# This is the final output of the llm call
print(result.final_output)

Why don’t Norwegians ever get lost?

Because they always fjord their way back.


In [6]:
# details of LLM call
input_list = result.to_input_list()

In [7]:
input_list

[{'content': 'Tell a joke about norwegians', 'role': 'user'},
 {'id': 'msg_0822e9f54217e15a006a709da2824c81929a00e5e61b8c11ab',
  'content': [{'annotations': [],
    'text': 'Why don’t Norwegians ever get lost?\n\nBecause they always fjord their way back.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [8]:
# Here are each of the steps
for index, each_step in enumerate(input_list, start=1):
    print(f"Step {index}: {each_step}")

Step 1: {'content': 'Tell a joke about norwegians', 'role': 'user'}
Step 2: {'id': 'msg_0822e9f54217e15a006a709da2824c81929a00e5e61b8c11ab', 'content': [{'annotations': [], 'text': 'Why don’t Norwegians ever get lost?\n\nBecause they always fjord their way back.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}


## Adding observability with trace

In [9]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about autonomous AI agents")
    print(result.final_output)

Why did the autonomous AI agent get promoted?

Because it finally stopped waiting for instructions and started making its own mistakes.


In [10]:
# streaming back the response
result = Runner.run_streamed(agent, "Tell med 5 jokes about Sweden")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
        

Sure — here are 5 Sweden jokes:

1. **Why don’t Swedes ever get lost?**  
   Because they always have a *Vasa* map.

2. **What do you call a Swedish dad joke?**  
   A *Nordic* punchline.

3. **Why did the Swedish chef bring a ladder?**  
   To reach the *high fjords*.

4. **What’s Sweden’s favorite type of music?**  
   *ABBA-cappella*.

5. **Why do Swedes make great friends?**  
   Because they’re always *lagom*—just the right amount of everything.

Want 5 more, but sillier?

## Part 2 - Add tools

In [11]:
# Create a function that record emails - A tool
def record_email(email_address):
    with open("./emails.txt", "a", encoding="utf-8") as f:
        f.write(email_address + "\n")
    return "Email received"

In [12]:
record_email("bakabha@gmail.com")

'Email received'

In [13]:
# Add toll using OpenAI SDK
@function_tool
def rec_email(email_address: str) -> str:
    """Record the user's email address"""
    with open("./emails.txt", "a", encoding="utf-8") as f:
        f.write(email_address + "\n")
    return f"Email {email_address} recorded successfully"
    
    

In [14]:
rec_email.params_json_schema

{'properties': {'email_address': {'title': 'Email Address', 'type': 'string'}},
 'required': ['email_address'],
 'title': 'rec_email_args',
 'type': 'object',
 'additionalProperties': False}

In [15]:
rec_email.description

"Record the user's email address"

In [17]:
# Create new agent
notifier_agent = Agent(name="Notifier", 
                       instructions="You're an assistant that record emails", 
                       model="gpt-5.4-mini",
                       tools=[rec_email])

In [18]:
with trace("Recording email"):
    result = await Runner.run(notifier_agent, "Please record the following email: 1212@gmail.com")
    print(result.final_output)
    

Recorded successfully.
